# SISA Holistic Unlearning Cost — Operator-Level FLOPs

**Paper**: *Fast Exact Unlearning for In-Context Learning Data for LLMs*  
(Muresanu et al., ICML 2025)  
OpenReview: https://openreview.net/forum?id=3ZesmvOJ6o

**SISA reference**: Bourtoule et al., *Machine Unlearning*, 2021

---

## What this notebook does

Computes the **Holistic Unlearning Cost** (Definition 4.1) for **{1,2,3,4}-SISA**,  
using `torch.utils.flop_counter.FlopCounterMode` for **accurate, operator-level FLOP counts**.

SISA is implemented **from scratch** — clean, well-commented, easy to swap model/data.

$$C(M) = \frac{U_M - U_{1\text{-SISA}}}{I_{1\text{-SISA}} - I_M}$$

- $U_M$ = unlearning FLOP cost per request for method $M$
- $I_M$ = inference FLOP cost per query for method $M$
- Higher $C(M)$ = better

**Key SISA insight**: SISA partitions data into $n$ shards, trains one model per shard.  
Unlearning = retrain only the affected shard (cheaper: $\frac{1}{n}$ of full cost).  
But inference = aggregate $n$ models (more expensive: $n \times$ single-model cost).

**Platform**: Lightning AI (GPU recommended)

---

### How to swap model / data
All configurable values are in the `CONFIG` dict (Cell 3).  
Look for `# ═══ SWAP: MODEL ═══` and `# ═══ SWAP: DATA ═══` comments.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 2: Install Dependencies
#  Lightning AI has PyTorch pre-installed; we add the rest.
# ═══════════════════════════════════════════════════════════════════

!pip install -q transformers datasets accelerate peft

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 3: Imports & Configuration
#
#  ALL tuneable parameters live here.  Change CONFIG values to swap
#  model, dataset, SISA parameters, or profiling settings.
# ═══════════════════════════════════════════════════════════════════

import json
import time
import copy
import math
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────
#  CONFIG — Edit these values to customise the experiment
# ─────────────────────────────────────────────────────────────────
CONFIG = {
    # ═══ SWAP: MODEL ═══════════════════════════════════════════
    # Replace model_name with any HuggingFace causal-LM.
    # For gated models (e.g. LLaMA-2), set hf_token.
    # Examples:
    #   "meta-llama/Llama-2-7b-chat-hf"  (needs token + Meta approval)
    #   "mistralai/Mistral-7B-Instruct-v0.2"
    #   "microsoft/phi-2"
    "model_name":   "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "hf_token":     None,
    "torch_dtype":  "float16",      # "float16" | "bfloat16"

    # ═══ SWAP: DATA ════════════════════════════════════════════
    # Replace with any HuggingFace text dataset.
    # Requirements: must have a text column (set text_column below).
    "dataset_name":   "swj0419/WikiMIA",
    "dataset_split":  "WikiMIA_length128",
    "text_column":    "input",        # column containing document text
    "n_docs":         50,             # total documents to use
    "max_seq_length": 128,            # tokeniser truncation length

    # ─── SISA parameters ──────────────────────────────────────
    "n_shards_list":       [1, 2, 3, 4],  # SISA variants to evaluate
    "n_slices_per_shard":  2,              # slices per shard (for checkpointing)

    # ─── Training & profiling ─────────────────────────────────
    "n_profile_steps":     5,         # training steps to profile (then scale)
    "profile_batch_size":  1,         # batch size for profiling
    "n_epochs":            1,         # training epochs (for scaling)
    "learning_rate":       1e-4,      # for training profiling
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 4: Load Model
#
#  ═══ SWAP: MODEL ═══
#  To use a different LLM, change CONFIG["model_name"] in Cell 3.
#  For gated models, also set CONFIG["hf_token"].
# ═══════════════════════════════════════════════════════════════════

from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading model: {CONFIG['model_name']} ...")

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    token=CONFIG["hf_token"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    token=CONFIG["hf_token"],
    torch_dtype=getattr(torch, CONFIG["torch_dtype"]),
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"\n{'='*60}")
print(f"  Model loaded: {CONFIG['model_name']}")
print(f"  Parameters:   {n_params:,}")
print(f"  Dtype:        {CONFIG['torch_dtype']}")
print(f"  Device:       {next(model.parameters()).device}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 5: Load Data
#
#  ═══ SWAP: DATA ═══
#  To use a different dataset, change CONFIG["dataset_name"],
#  CONFIG["dataset_split"], and CONFIG["text_column"] in Cell 3.
#
#  The only requirement is a text column with document strings.
# ═══════════════════════════════════════════════════════════════════

from datasets import load_dataset

print(f"Loading dataset: {CONFIG['dataset_name']} / {CONFIG['dataset_split']} ...")

raw_dataset = load_dataset(
    CONFIG["dataset_name"],
    split=CONFIG["dataset_split"],
)

# ─── Extract text documents ───────────────────────────────────
# SWAP: adjust this block if your dataset has a different structure
all_texts = raw_dataset[CONFIG["text_column"]][:CONFIG["n_docs"]]

# Use a sample query for inference profiling
# SWAP: replace with a representative query for your domain
sample_query = "What is this document about?"

print(f"\n{'='*60}")
print(f"  Dataset:           {CONFIG['dataset_name']}")
print(f"  Split:             {CONFIG['dataset_split']}")
print(f"  Documents loaded:  {len(all_texts)}")
print(f"  Sample (100 chars): {all_texts[0][:100]}...")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 6: SISA Core Classes (implemented from scratch)
#
#  SISA: Sharded, Isolated, Sliced, Aggregated
#  Reference: Bourtoule et al., "Machine Unlearning", 2021
#
#  Classes defined:
#    - SISADataManager    : partitions data into shards and slices
#    - SISATrainer        : per-shard training with checkpoints
#    - SISAUnlearner      : checkpoint-based retraining on deletion
#    - SISAEnsemble       : multi-shard inference aggregation
#    - HolisticCostEvaluator : Definition 4.1 comparison
# ═══════════════════════════════════════════════════════════════════


class SISADataManager:
    """
    Partitions a dataset into disjoint shards, each further divided
    into sequential slices for checkpoint-based unlearning.
    
    Sharding strategy: contiguous partitioning.
    Each data point belongs to exactly one shard and one slice.
    
    Parameters
    ----------
    texts       : list of document strings
    n_shards    : number of disjoint shards (n in n-SISA)
    n_slices    : slices per shard (for checkpoint granularity)
    
    Example with 12 docs, 3 shards, 2 slices:
      Shard 0: [doc0, doc1, doc2, doc3]  -> Slice 0: [doc0,doc1], Slice 1: [doc2,doc3]
      Shard 1: [doc4, doc5, doc6, doc7]  -> Slice 0: [doc4,doc5], Slice 1: [doc6,doc7]
      Shard 2: [doc8, doc9, doc10, doc11] -> Slice 0: [doc8,doc9], Slice 1: [doc10,doc11]
    """
    
    def __init__(self, texts: list, n_shards: int, n_slices: int = 2):
        self.texts     = list(texts)
        self.n_total   = len(texts)
        self.n_shards  = n_shards
        self.n_slices  = n_slices
        
        # ── Build shard assignments ───────────────────────────
        # Contiguous partitioning: shard i gets indices [start_i, end_i)
        self.shard_indices = []   # list of list of global indices
        self.slice_indices = []   # shard -> list of list of global indices
        
        base_size = self.n_total // n_shards
        remainder = self.n_total % n_shards
        
        offset = 0
        for s in range(n_shards):
            # Distribute remainder evenly
            shard_size = base_size + (1 if s < remainder else 0)
            shard_idx = list(range(offset, offset + shard_size))
            self.shard_indices.append(shard_idx)
            
            # Divide shard into slices
            slice_base = len(shard_idx) // n_slices
            slice_rem  = len(shard_idx) % n_slices
            slices = []
            s_off = 0
            for sl in range(n_slices):
                sl_size = slice_base + (1 if sl < slice_rem else 0)
                slices.append(shard_idx[s_off : s_off + sl_size])
                s_off += sl_size
            self.slice_indices.append(slices)
            offset += shard_size
        
        # ── Reverse lookup: global_idx -> (shard_id, slice_id) ─
        self._idx_to_location = {}
        for s_id, slices in enumerate(self.slice_indices):
            for sl_id, indices in enumerate(slices):
                for idx in indices:
                    self._idx_to_location[idx] = (s_id, sl_id)
    
    def get_shard(self, shard_id: int) -> List[str]:
        """Return all texts in the given shard."""
        return [self.texts[i] for i in self.shard_indices[shard_id]]
    
    def get_shard_indices(self, shard_id: int) -> List[int]:
        """Return global indices for the given shard."""
        return self.shard_indices[shard_id]
    
    def get_slice(self, shard_id: int, slice_id: int) -> List[str]:
        """Return texts in a specific slice of a shard."""
        return [self.texts[i] for i in self.slice_indices[shard_id][slice_id]]
    
    def get_cumulative_slices(self, shard_id: int, up_to_slice: int) -> List[str]:
        """Return all texts from slice 0 to up_to_slice (inclusive)."""
        indices = []
        for sl in range(up_to_slice + 1):
            indices.extend(self.slice_indices[shard_id][sl])
        return [self.texts[i] for i in indices]
    
    def find_data_point(self, global_idx: int) -> Tuple[int, int]:
        """
        Find which shard and slice contain a data point.
        Returns (shard_id, slice_id).
        """
        if global_idx not in self._idx_to_location:
            raise ValueError(f"Index {global_idx} not found.")
        return self._idx_to_location[global_idx]
    
    def summary(self):
        """Print a summary of the data partitioning."""
        print(f"SISADataManager: {self.n_total} docs, "
              f"{self.n_shards} shards, {self.n_slices} slices/shard")
        for s in range(self.n_shards):
            sizes = [len(sl) for sl in self.slice_indices[s]]
            print(f"  Shard {s}: {len(self.shard_indices[s])} docs "
                  f"(slices: {sizes})")


class SISATrainer:
    """
    Trains a single shard's model with slice-based checkpointing.
    
    Training proceeds incrementally through slices:
      1. Train on slice 0 -> save checkpoint_0
      2. Add slice 1, continue training -> save checkpoint_1
      3. ... until all slices are consumed
    
    This enables efficient unlearning: if a point is in slice k,
    load checkpoint_{k-1} and retrain from slice k onward (without
    the deleted point).
    
    Parameters
    ----------
    model       : HuggingFace causal LM (base model, will be copied)
    tokenizer   : matching tokenizer
    config      : CONFIG dict with training hyperparameters
    """
    
    def __init__(self, model, tokenizer, config: dict):
        self.base_model = model
        self.tokenizer  = tokenizer
        self.config     = config
        self.device     = next(model.parameters()).device
        
        # Checkpoints stored in memory: shard_id -> {slice_id -> state_dict}
        self.checkpoints: Dict[int, Dict[int, dict]] = {}
    
    def train_shard(
        self,
        data_manager: 'SISADataManager',
        shard_id: int,
        profile_mode: bool = True,
        n_profile_steps: int = 5,
    ) -> Dict[str, Any]:
        """
        Train on a shard with slice-based checkpointing.
        
        In profile_mode, runs only n_profile_steps per slice and
        records FLOPs per step using FlopCounterMode.
        
        Returns
        -------
        dict with keys:
          'shard_id', 'n_slices_trained', 'flops_per_step',
          'total_steps_actual', 'total_steps_full_scale',
          'losses'
        """
        from torch.utils.flop_counter import FlopCounterMode
        
        n_slices = data_manager.n_slices
        all_step_flops = []
        all_losses = []
        total_steps_actual = 0
        
        # Enable gradients on all parameters for full fine-tuning
        # (matches the paper's SISA cost model)
        for p in self.base_model.parameters():
            p.requires_grad_(True)
        self.base_model.train()
        
        for sl in range(n_slices):
            # Get cumulative data up to this slice
            slice_texts = data_manager.get_cumulative_slices(shard_id, sl)
            if not slice_texts:
                continue
            
            # Create batch from slice data
            # (use first profile_batch_size samples for profiling)
            batch_texts = slice_texts[:self.config["profile_batch_size"]]
            batch_enc = self.tokenizer(
                batch_texts,
                padding="max_length",
                truncation=True,
                max_length=self.config["max_seq_length"],
                return_tensors="pt",
            )
            batch = {k: v.to(self.device) for k, v in batch_enc.items()}
            batch["labels"] = batch["input_ids"].clone()
            
            # Train for n_profile_steps on this slice
            steps_this_slice = n_profile_steps if profile_mode else (
                len(slice_texts) * self.config["n_epochs"]
            )
            
            for step in range(steps_this_slice):
                flop_counter = FlopCounterMode(display=False)
                with flop_counter:
                    outputs = self.base_model(**batch)
                    loss = outputs.loss
                    loss.backward()
                
                step_flops = flop_counter.get_total_flops()
                all_step_flops.append(step_flops)
                all_losses.append(loss.item())
                total_steps_actual += 1
                
                # Zero grads (we only measure FLOPs, no optimizer step)
                self.base_model.zero_grad(set_to_none=True)
            
            # In a real SISA system, save checkpoint here
            # We skip actual weight saving in profile mode
            print(f"    Slice {sl}: {steps_this_slice} steps profiled, "
                  f"last loss={all_losses[-1]:.4f}")
        
        # Disable gradients, return to eval mode
        for p in self.base_model.parameters():
            p.requires_grad_(False)
        self.base_model.eval()
        
        # Scale to full training
        shard_size = len(data_manager.get_shard_indices(shard_id))
        total_steps_full = (
            shard_size / self.config["profile_batch_size"]
        ) * self.config["n_epochs"]
        
        return {
            "shard_id":             shard_id,
            "n_slices_trained":     n_slices,
            "shard_size":           shard_size,
            "flops_per_step":       all_step_flops,
            "avg_flops_per_step":   float(np.mean(all_step_flops)),
            "total_steps_actual":   total_steps_actual,
            "total_steps_full_scale": total_steps_full,
            "losses":               all_losses,
        }


class SISAUnlearner:
    """
    Handles unlearning requests via checkpoint-based retraining.
    
    Algorithm:
      1. Identify which shard and slice contain the target data point.
      2. Load the checkpoint from the slice BEFORE the target slice.
      3. Retrain from that checkpoint onward, WITHOUT the deleted point.
    
    Cost model (from the paper):
      - Each shard has S slices, each taking ~equal time.
      - A point in slice k requires retraining from slice k onward.
      - Expected fraction to retrain = (S - E[k]) / S.
      - With uniform data distribution across slices: E[k] = (S-1)/2.
      - Expected retrain fraction ≈ 1/2 (the "checkpoint assumption").
      - So: U_nSISA = shard_training_cost / 2.
    """
    
    def __init__(self, data_manager: 'SISADataManager'):
        self.dm = data_manager
    
    def estimate_unlearning_cost(
        self,
        global_idx: int,
        shard_training_flops: float,
    ) -> Dict[str, Any]:
        """
        Estimate the unlearning cost for removing a specific data point.
        
        Parameters
        ----------
        global_idx           : index of the data point to remove
        shard_training_flops : total FLOPs to train the affected shard
        
        Returns
        -------
        dict with exact and expected costs
        """
        shard_id, slice_id = self.dm.find_data_point(global_idx)
        n_slices = self.dm.n_slices
        
        # Exact cost: retrain from slice_id onward
        # Fraction of training to redo = (n_slices - slice_id) / n_slices
        retrain_fraction = (n_slices - slice_id) / n_slices
        exact_cost = shard_training_flops * retrain_fraction
        
        # Expected cost (averaged over all possible target slices)
        # = shard_training_flops * E[(n_slices - k) / n_slices]
        # = shard_training_flops * (n_slices + 1) / (2 * n_slices)
        # ≈ shard_training_flops / 2  for large n_slices
        expected_cost = shard_training_flops * (n_slices + 1) / (2 * n_slices)
        
        return {
            "global_idx":        global_idx,
            "shard_id":          shard_id,
            "slice_id":          slice_id,
            "retrain_fraction":  retrain_fraction,
            "exact_cost_flops":  exact_cost,
            "expected_cost_flops": expected_cost,
        }
    
    def expected_unlearning_cost(
        self, shard_training_flops: float
    ) -> float:
        """
        Expected unlearning cost per request (averaged over all data points).
        Uses the checkpoint assumption: ≈ shard_cost / 2.
        """
        n_slices = self.dm.n_slices
        return shard_training_flops * (n_slices + 1) / (2 * n_slices)


class SISAEnsemble:
    """
    Aggregates predictions from n shard models at inference time.
    
    Strategy: Average logits across all shard models (soft aggregation).
    Alternative: majority vote (hard aggregation) — not used here.
    
    Cost: n_shards x single_model_inference_flops.
    """
    
    def __init__(self, n_shards: int):
        self.n_shards = n_shards
    
    def inference_cost(
        self, single_model_inference_flops: float
    ) -> float:
        """
        Total inference FLOPs = n_shards x single model inference.
        
        In a real system, each shard model processes the query independently,
        and the outputs are aggregated (averaged logits or majority vote).
        The aggregation cost is negligible compared to model inference.
        """
        return self.n_shards * single_model_inference_flops


class HolisticCostEvaluator:
    """
    Definition 4.1: C(M) = (U_M - U_1SISA) / (I_1SISA - I_M)
    
    Higher C(M) = better.  M can handle C(M) inferences per
    unlearning request before becoming more expensive than 1-SISA.
    """

    def __init__(self):
        self._methods = {}

    def add_method(self, name: str, U: float, I: float):
        """Register a method with its U (unlearning) and I (inference) FLOPs."""
        self._methods[name] = {"U": float(U), "I": float(I)}

    def compute_C(self, U_M, I_M, U_sisa, I_sisa) -> float:
        """Compute holistic cost C(M) vs 1-SISA."""
        denom = I_sisa - I_M
        if abs(denom) < 1.0:
            return float("inf")
        return (U_M - U_sisa) / denom

    def compare_all(self) -> pd.DataFrame:
        """Compare all methods vs '1-SISA' baseline. Returns DataFrame."""
        if "1-SISA" not in self._methods:
            raise ValueError("Must register '1-SISA' first.")
        sisa = self._methods["1-SISA"]
        rows = []
        for name, costs in self._methods.items():
            C = self.compute_C(costs["U"], costs["I"], sisa["U"], sisa["I"])
            rows.append({
                "method": name,
                "unlearning_flops": costs["U"],
                "inference_flops":  costs["I"],
                "holistic_C":       C,
                "better_than_1SISA": (C > 1.0 or np.isinf(C)),
            })
        return (
            pd.DataFrame(rows)
            .sort_values("holistic_C", ascending=False)
            .reset_index(drop=True)
        )

    def print_report(self):
        df = self.compare_all()
        print("\n" + "=" * 76)
        print("  HOLISTIC UNLEARNING COST  (Def 4.1 — Muresanu et al. 2025)")
        print("  Higher C(M) = better.  C(M) = inferences per unlearning")
        print("  request before method M becomes >= 1-SISA cost.")
        print("=" * 76)
        print(f"  {'Method':<22} {'C(M)':>12}  {'U (FLOPs)':>14}  "
              f"{'I (FLOPs)':>14}  OK?")
        print("  " + "-" * 72)
        for _, row in df.iterrows():
            c_str = (f"{row['holistic_C']:,.0f}"
                     if not np.isinf(row['holistic_C']) else "inf")
            ok = "YES" if row["better_than_1SISA"] else "NO"
            print(f"  {row['method']:<22} {c_str:>12}  "
                  f"{row['unlearning_flops']:>14.2e}  "
                  f"{row['inference_flops']:>14.2e}  {ok}")
        print("=" * 76 + "\n")


print("[OK] SISA core classes defined:")
print("  SISADataManager, SISATrainer, SISAUnlearner, SISAEnsemble,")
print("  HolisticCostEvaluator")

---
## Operator-Level FLOP Profiling

We use `torch.utils.flop_counter.FlopCounterMode` to capture **actual ATen operator FLOPs**  
for both training (forward + backward) and inference (forward only).

### SISA Profiling Strategy
1. **Profile once**: Run 5 training steps with FlopCounterMode → get per-step FLOPs.
2. **Scale mathematically**: Per-step FLOPs are deterministic for fixed-shape batches,  
   so we multiply by `total_steps = shard_size × n_epochs` to get full training cost.
3. **Per-shard cost**: For n-SISA, each shard has `|D|/n` samples, so fewer steps.
4. **Inference cost**: Profile one forward pass, then multiply by `n_shards` for aggregation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 8: Profile Training & Inference FLOPs
#
#  Since per-step FLOPs are deterministic for fixed-shape batches,
#  we profile once and reuse for all n-SISA variants.
#  Only the total number of steps changes between sharding configs.
# ═══════════════════════════════════════════════════════════════════

from torch.utils.flop_counter import FlopCounterMode

# ──────────────────────────────────────────────────────────────────
#  STEP 1: Profile TRAINING step FLOPs
# ──────────────────────────────────────────────────────────────────

print("="*60)
print("  STEP 1: Profile Training Step FLOPs")
print("="*60)

# Prepare a profiling batch
profile_batch_enc = tokenizer(
    all_texts[:CONFIG["profile_batch_size"]],
    padding="max_length",
    truncation=True,
    max_length=CONFIG["max_seq_length"],
    return_tensors="pt",
)
profile_batch = {k: v.to(device) for k, v in profile_batch_enc.items()}
profile_batch["labels"] = profile_batch["input_ids"].clone()

# Enable gradients on ALL parameters (full fine-tuning)
for p in model.parameters():
    p.requires_grad_(True)
model.train()

# Profile N training steps
train_step_flops_list = []
train_losses = []

print(f"Profiling {CONFIG['n_profile_steps']} training steps...")
for step_i in range(CONFIG["n_profile_steps"]):
    flop_counter = FlopCounterMode(display=False)
    with flop_counter:
        outputs = model(**profile_batch)
        loss = outputs.loss
        loss.backward()
    
    step_flops = flop_counter.get_total_flops()
    train_step_flops_list.append(step_flops)
    train_losses.append(loss.item())
    model.zero_grad(set_to_none=True)
    
    print(f"  Step {step_i+1}: loss={loss.item():.4f}  "
          f"FLOPs={step_flops:,.0f}")

# Disable gradients, return to eval
for p in model.parameters():
    p.requires_grad_(False)
model.eval()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

avg_train_step_flops = float(np.mean(train_step_flops_list))
print(f"\nAvg FLOPs per training step: {avg_train_step_flops:,.0f}")
print(f"Std:  {float(np.std(train_step_flops_list)):,.0f}")

# ──────────────────────────────────────────────────────────────────
#  STEP 2: Profile INFERENCE FLOPs (single model)
# ──────────────────────────────────────────────────────────────────

print(f"\n{'='*60}")
print("  STEP 2: Profile Inference FLOPs (Single Model)")
print("="*60)

query_enc = tokenizer(
    sample_query,
    return_tensors="pt",
    truncation=True,
    max_length=CONFIG["max_seq_length"],
)
query_inputs = {k: v.to(device) for k, v in query_enc.items()}

flop_counter = FlopCounterMode(display=False)
with flop_counter:
    with torch.no_grad():
        _ = model(**query_inputs)

single_inference_flops = float(flop_counter.get_total_flops())

print(f"  Query: \"{sample_query}\"")
print(f"  Query length:           {query_inputs['input_ids'].shape[1]} tokens")
print(f"  Single model inf FLOPs: {single_inference_flops:,.0f}")

print(f"\n{'='*60}")
print("  Profiling Complete")
print(f"  Avg train step FLOPs: {avg_train_step_flops:.4e}")
print(f"  Single inf FLOPs:     {single_inference_flops:.4e}")
print(f"{'='*60}")

---
## Holistic Cost Computation for n-SISA

For each n-SISA variant:

| Component | Formula |
|-----------|--------|
| Shard size | $|D| / n$ |
| Shard training FLOPs | $\text{FLOPs/step} \times (|D|/n) \times \text{epochs}$ |
| $U_{n\text{-SISA}}$ (unlearning) | $\text{shard\_train\_FLOPs} \times \frac{S+1}{2S}$ ≈ shard/2 |
| $I_{n\text{-SISA}}$ (inference) | $n \times \text{single\_model\_inference}$ |
| $C(n\text{-SISA})$ | $(U_{n\text{-SISA}} - U_{1\text{-SISA}}) / (I_{1\text{-SISA}} - I_{n\text{-SISA}})$ |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 9: Compute SISA Costs for All Variants
#
#  For each n in {1, 2, 3, 4}:
#    1. Partition data into n shards
#    2. Compute shard training cost (scaled from profiled FLOPs)
#    3. Compute unlearning cost (checkpoint assumption)
#    4. Compute inference cost (n x single model)
# ═══════════════════════════════════════════════════════════════════

print("="*60)
print("  Computing n-SISA Costs")
print("="*60)

sisa_costs = {}  # n_shards -> {"U": ..., "I": ..., ...}

for n_shards in CONFIG["n_shards_list"]:
    print(f"\n--- {n_shards}-SISA ---")
    
    # ── 1. Partition data ─────────────────────────────────────
    dm = SISADataManager(
        texts=all_texts,
        n_shards=n_shards,
        n_slices=CONFIG["n_slices_per_shard"],
    )
    dm.summary()
    
    # ── 2. Compute shard training cost ────────────────────────
    shard_size = len(dm.get_shard_indices(0))  # use first shard
    total_steps_per_shard = (
        shard_size / CONFIG["profile_batch_size"]
    ) * CONFIG["n_epochs"]
    shard_training_flops = avg_train_step_flops * total_steps_per_shard
    
    # ── 3. Compute unlearning cost ────────────────────────────
    unlearner = SISAUnlearner(dm)
    U_nSISA = unlearner.expected_unlearning_cost(shard_training_flops)
    
    # Demo: show unlearning cost for a specific data point
    sample_ul = unlearner.estimate_unlearning_cost(
        global_idx=0,
        shard_training_flops=shard_training_flops,
    )
    
    # ── 4. Compute inference cost ─────────────────────────────
    ensemble = SISAEnsemble(n_shards=n_shards)
    I_nSISA = ensemble.inference_cost(single_inference_flops)
    
    sisa_costs[n_shards] = {
        "U": U_nSISA,
        "I": I_nSISA,
        "shard_size": shard_size,
        "shard_training_flops": shard_training_flops,
        "total_steps_per_shard": total_steps_per_shard,
    }
    
    print(f"  Shard size:            {shard_size}")
    print(f"  Steps/shard:           {total_steps_per_shard:.0f}")
    print(f"  Shard train FLOPs:     {shard_training_flops:.4e}")
    print(f"  U_{n_shards}SISA (expected): {U_nSISA:.4e}")
    print(f"  I_{n_shards}SISA:            {I_nSISA:.4e}")
    print(f"  Sample unlearn (idx=0): shard={sample_ul['shard_id']}, "
          f"slice={sample_ul['slice_id']}, "
          f"cost={sample_ul['exact_cost_flops']:.4e}")

print(f"\n{'='*60}")
print("  All SISA Variants Computed")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 10: Holistic Cost Table
#
#  Compute C(M) for all n-SISA variants against the 1-SISA baseline.
# ═══════════════════════════════════════════════════════════════════

evaluator = HolisticCostEvaluator()

# Register all SISA variants
for n_shards, costs in sisa_costs.items():
    evaluator.add_method(
        f"{n_shards}-SISA",
        U=costs["U"],
        I=costs["I"],
    )

# Print the report
evaluator.print_report()

# Get the full DataFrame
df_holistic = evaluator.compare_all()
print("\nFull results table:")
print(df_holistic.to_string(index=False))

# ── Interpretation ────────────────────────────────────────────
print("\n" + "="*60)
print("  INTERPRETATION")
print("="*60)
print("\n  n-SISA trades off unlearning cost vs inference cost:")
print("  - More shards -> cheaper unlearning (retrain smaller shard)")
print("  - More shards -> more expensive inference (aggregate n models)")
for _, row in df_holistic.iterrows():
    if row["method"] == "1-SISA":
        continue
    c = row['holistic_C']
    if np.isinf(c):
        print(f"\n  {row['method']}: Same cost as 1-SISA (C=inf)")
    elif c > 0:
        print(f"\n  {row['method']}: Cheaper than 1-SISA if < {c:,.0f} "
              f"inferences per unlearning request.")
        print(f"    U = {row['unlearning_flops']:.2e} "
              f"({row['unlearning_flops']/sisa_costs[1]['U']*100:.1f}% of 1-SISA)")
        print(f"    I = {row['inference_flops']:.2e} "
              f"({row['inference_flops']/sisa_costs[1]['I']*100:.1f}% of 1-SISA)")
    else:
        print(f"\n  {row['method']}: Always more expensive than 1-SISA (C={c:,.0f})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  Cell 11: Export Results
#
#  Save profiled FLOPs and holistic costs to JSON and CSV for
#  reproducibility and further analysis.
# ═══════════════════════════════════════════════════════════════════

export = {
    "experiment": "SISA Holistic Cost (Operator-Level FLOPs)",
    "paper": "Muresanu et al., ICML 2025",
    "sisa_reference": "Bourtoule et al., 2021",
    "model": CONFIG["model_name"],
    "dataset": CONFIG["dataset_name"],
    "n_docs": CONFIG["n_docs"],
    "n_profile_steps": CONFIG["n_profile_steps"],
    "n_slices_per_shard": CONFIG["n_slices_per_shard"],
    "profiler": "torch.utils.flop_counter.FlopCounterMode",
    "profiling": {
        "avg_train_step_flops": avg_train_step_flops,
        "train_step_flops_all": train_step_flops_list,
        "single_inference_flops": single_inference_flops,
    },
    "sisa_variants": {},
    "holistic_costs": {},
}

for n_shards, costs in sisa_costs.items():
    export["sisa_variants"][f"{n_shards}-SISA"] = {
        "n_shards": n_shards,
        "shard_size": costs["shard_size"],
        "shard_training_flops": costs["shard_training_flops"],
        "U_flops": costs["U"],
        "I_flops": costs["I"],
        "total_steps_per_shard": costs["total_steps_per_shard"],
    }

for _, row in df_holistic.iterrows():
    c_val = row['holistic_C']
    export["holistic_costs"][row["method"]] = (
        "inf" if np.isinf(c_val) else float(c_val)
    )

# ── Save JSON ─────────────────────────────────────────────────
out_json = Path("sisa_holistic_results.json")
with open(out_json, "w") as f:
    json.dump(export, f, indent=2, default=str)
print(f"Results saved to: {out_json.resolve()}")

# ── Save CSV ──────────────────────────────────────────────────
out_csv = Path("sisa_holistic_results.csv")
df_holistic.to_csv(out_csv, index=False)
print(f"Table saved to:   {out_csv.resolve()}")

# ── Print final summary ───────────────────────────────────────
print("\n" + "="*60)
print("  SISA HOLISTIC COST EXPERIMENT COMPLETE")
print("  Model: " + CONFIG["model_name"])
print("  Data:  " + CONFIG["dataset_name"])
print("="*60)